# 08 · Единый вход больничного бланка и границы UCI

## tl;dr

У приложения один публичный контракт — src.features.ctg_report_contract. Отсутствующее поле и неизвестное определение останавливают оценку до predict_proba. SisPorto-поля UCI остаются отдельным native benchmark import; STV(ms) не приравнивается к MSTV/ASTV.


## Context & Methods
### Key Assumptions
Числа примера переписаны с изображения; OCR не выполняется. Строка «8 из 8 в норме» сохраняется как заключение, не становится target или признаком. Лабораторные единицы и происхождение измерений проверяются отдельно. Полный аудит JNU — в [отдельной серии](../jnu_ctg/01_prepare_data.ipynb).


In [3]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
REVIEW=ROOT/'research/reviews/ctg_20261017'
UCI=ROOT/'research/experiments/ctg_uci/reviewed_v3'
JNU=ROOT/'research/experiments/jnu_ctg/reviewed_v2'
FIGURES=REVIEW/'figures';FIGURES.mkdir(exist_ok=True)
pd.set_option('display.max_colwidth',170)
sns.set_theme(style='whitegrid',font_scale=1.0)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False})
COLORS={'linear':'#009e73','spline':'#d55e00','catboost':'#0072b2'}
def save_show(name):
    plt.gcf().savefig(FIGURES/name,dpi=140,bbox_inches='tight')
    plt.show()


## Data · единый публичный вход

In [5]:
from src.features.ctg_russian_reports import EXAMPLE_TEXT
from src.features.ctg_report_contract import parse_ctg_report,prepare_report,evaluate_report,partial_dr_checklist,PROFILES
parsed=parse_ctg_report(EXAMPLE_TEXT);report=parsed['measurements']
display(pd.DataFrame(parsed['evidence']))
x,audit=prepare_report(report,'report5')
display(x);display(pd.DataFrame({'blocker':audit['blockers']}))
class MustNotBeCalled:
    def predict_proba(self,X):raise AssertionError('Blocked report reached model')
result=evaluate_report(report,MustNotBeCalled(),'report5')
assert result['status']=='abstained' and result['probability_expert_abnormal'] is None
display(pd.Series(result).to_frame('application response'))


,field,value,line
0,duration_min,17.0,"Время анализа 17,0 (10–60) мин"
1,signal_loss_pct,8.5,"Потери сигнала 8,5 %"
2,baseline_bpm,131.4,"Базальная ЧСС 131,4 (115–160) BPM"
3,acc10_count,5.0,Акцелерации >10 BPM и 15 с 5
4,acc15_count,1.0,Акцелерации >15 BPM и 15 с ✓ 1 (≥1)
5,dec15_count,0.0,Децелерации >15 BPM и 15 с 0
6,dec20lost_count,0.0,Децелерации >20 ударов ✓ 0 (≤0)
7,high_variation_min,13.0,Высокая вариабельность ✓ 13 (≥3) мин
8,low_variation_min,0.0,Низкая вариабельность 0 мин
9,sinusoidal_min,0.0,Синусоидальный ритм ✓ 0 (≤0) мин


,baseline_bpm,acc15_per_10min,dec15_per_10min,stv_ms,amplitude_bpm
0,131.4,0.588235,0.0,7.6,14.1


,blocker
0,transcription_unconfirmed
1,antenatal_context_unconfirmed
2,duration_not_validated_30min
3,measurement_bridge_unconfirmed
4,stv_3_75s_recipe_unconfirmed
5,amplitude_recipe_unconfirmed
6,event_denominator_unconfirmed


,application response
status,abstained
profile,report5
reasons,"[transcription_unconfirmed, antenatal_context_unconfirmed, duration_not_validated_30min, measurement_bridge_unconfirmed, stv_3_75s_recipe_unconfirmed, amplitude_recip..."
probability_expert_abnormal,None
clinical_risk,None
supported_for_clinical_inference,False


## Results · что известно и чего не хватает

In [7]:
display(pd.DataFrame({'profile':list(PROFILES),'fields':[', '.join(v) for v in PROFILES.values()]}))
check=partial_dr_checklist(report);display(pd.DataFrame(check['requirements']))
assert check['criteria_met'] is None
mapping=pd.read_csv(ROOT/'research/experiments/ctg_uci/report_features_v1/hospital_feature_mapping.csv')
display(mapping)


,profile,fields
0,core3,"baseline_bpm, acc15_per_10min, dec15_per_10min"
1,report5,"baseline_bpm, acc15_per_10min, dec15_per_10min, stv_ms, amplitude_bpm"
2,report7,"baseline_bpm, acc15_per_10min, dec15_per_10min, stv_ms, amplitude_bpm, acc10_per_10min, dec20lost_per_10min"
3,report4s5,"baseline_bpm, acc15_per_10min, dec15_per_10min, stv_4s_ms, amplitude_bpm"


,requirement,status
0,STV >3 ms in the specified implementation,not_evaluable
1,Gestation-dependent high-variation LTV centiles,not_evaluable
2,Verified high-variation episode and sinusoidal assessment,not_evaluable
3,Movement/acceleration combination with verified event definitions,not_evaluable
4,Deceleration area categories including >100 lost beats,not_evaluable
5,Terminal epoch deceleration and artifact checks,not_evaluable


,hospital_field,UCI_candidate,status,reason
0,"Базальная ЧСС, BPM",LB,conditional,Единицы совпадают; алгоритм базальной линии и версия прибора требуют проверки
1,"Акцелерации >15 BPM, ≥15 s",AC,conditional,count/(60×duration_min); совпадения порога недостаточно для равенства детекторов
2,"Акцелерации >10 BPM, ≥15 s",нет,incompatible,Другой порог: не подставлять в AC и не складывать с >15
3,"Все децелерации >15 BPM, ≥15 s",DL+DS+DP,conditional_proxy,Сумма опубликованных категорий UCI; сверить взаимоисключение и правила детектора
4,Децелерации >20 потерянных ударов,нет,incompatible,"Площадь события, не амплитуда >20 BPM и не число DP"
5,"STV, ms / BPM",MSTV,incompatible,Эпохальная STV и исходная SisPorto MSTV не взаимозаменяемы
6,"ВКИ, ms / BPM",нет,incompatible,В руководстве Сономед это vSTV от удара к удару; не доказано равенство UCI MSTV
7,Минуты высокой / низкой вариабельности,ASTV / ALTV,incompatible,Эпизоды с иными правилами; длительность / время записи не восстанавливает ASTV/ALTV
8,Амплитуда осцилляций,Width / MLTV,incompatible,Минутная амплитуда не ширина общей гистограммы; MLTV требует нативного определения
9,"Частота осцилляций, 1/min",Nmax / Nzeros,incompatible,Частота во времени не число пиков/нулей гистограммы


## Takeaways
Для приложения используется один parser/контракт; UCI-поля не подставляются по сходству названий. Полный D/R или Фишер по неполному бланку не вычисляется. До подтверждения формул целевого устройства и независимой разметки модель не выдаёт медицинскую вероятность.
